**1. Set up and Load Data**

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px


transactions_df_spark=spark.read.table("mthimunye.default.freshmart_transactions")

stockouts_df_spark=spark.read.table("mthimunye.default.freshmart_stockouts")

stores_df_spark=spark.read.table("mthimunye.default.freshmart_stores")

**1.1 Convert pyspark dataframe into pandas dataframe**

In [0]:
transactions_df=transactions_df_spark.toPandas()
stockouts_df=stockouts_df_spark.toPandas()
stores_df=stores_df_spark.toPandas()

display(transactions_df.head(5))
display(stockouts_df.head(5))
display(stores_df.head(5))

**2. Dataset Interview**

**2.1 How many rows and columns are there**

In [0]:
transactions_df.shape

In [0]:
stockouts_df.shape

In [0]:
stores_df.shape

**2.2 What is the data type of each column** 

In [0]:
transactions_df.dtypes

In [0]:
stockouts_df.dtypes

In [0]:
stores_df.dtypes

**2.3 Statistical summary of numeric columns** 

In [0]:
transactions_df.describe()

In [0]:
stockouts_df.describe()

In [0]:
stores_df.describe()

**2.4 All in once- Row/column count, missing values and datatypes**

In [0]:
transactions_df.info()

In [0]:
stores_df.info()

In [0]:
stockouts_df.info()

**3. Clean and Prepare**

**3.1 Nulls or missing values inspection**

In [0]:

# We check missing values to ensure data completeness before performing analysis.
transactions_df.isnull().sum()


In [0]:
stores_df.isnull().sum()

In [0]:
stockouts_df.isnull().sum()

**Null Analysis Report**

1. transactions_df : customer_id : 35674 nulls
replace the nulls = non_loyalty

2. stores_df : competitor_open_date : 30
replace the nulls =

3. stockouts_df : 0 columns with nulls

In [0]:
# Convert transaction_date to datetime in transactions_df
transactions_df['transaction_date'] = pd.to_datetime(transactions_df['transaction_date'])

# Convert date to datetime in stockouts_df
stockouts_df['date'] = pd.to_datetime(stockouts_df['date'])

# Convert competitor_open_date to datetime in stores_df (handles nulls safely)
stores_df['competitor_open_date'] = pd.to_datetime(stores_df['competitor_open_date'])

# Verify the updated datatypes
print("Transactions date type:", transactions_df['transaction_date'].dtype)
print("Stockouts date type:", stockouts_df['date'].dtype)
print("Stores competitor date type:", stores_df['competitor_open_date'].dtype)

In [0]:
# Handle missing customer_id for non-loyalty members
transactions_df['customer_id'] = transactions_df['customer_id'].fillna('Non-Loyalty')
display(transactions_df.head(3))

In [0]:
# 1. Extract year-month for monthly trend analysis
transactions_df['month'] = transactions_df['transaction_date'].dt.to_period('M').astype(str)

# 2. Merge transactions with stores data on store_id
merged_df = transactions_df.merge(stores_df, on='store_id', how='left')

# 3. Filter/Group by Month to check the trend (Footfall vs Revenue)
monthly_trend = merged_df.groupby('month').agg(
    total_revenue=('basket_value_zar', 'sum'),
    transaction_count=('transaction_id', 'count'), # Footfall proxy
    avg_basket_value=('basket_value_zar', 'mean'),
    avg_items=('num_items', 'mean')
).reset_index()

display(monthly_trend)

In [0]:
#Filter and segment by Store Format and Competitor Presence

format_competitor_segment = merged_df.groupby(['store_format', 'has_nearby_competitor']).agg(
    total_revenue=('basket_value_zar', 'sum'),
    transaction_count=('transaction_id', 'count'),
    avg_basket_value=('basket_value_zar', 'mean')
).reset_index()

display(format_competitor_segment)

**The Core Paradox (Footfall vs. Basket Value):**

Looking at your monthly trend, transaction count (footfall proxy) actually increases from January (5,175 transactions) up through November and December (5,829 and 5,674 transactions), confirming leadership's note that footfall is up.

However, average basket value is steadily shrinking month-over-month—dropping significantly from ZAR 299.82 in January all the way down to ZAR 248.56 by December. Similarly, average items per basket fall from 14.75 to 12.06. This explains why total revenue stays stubbornly flat despite more people walking through the doors.

**Store Format & Competitor Impact:**

When looking at your segment analysis, stores with a nearby discounter (has_nearby_competitor == 'Yes') consistently suffer lower average basket values across every single format compared to those without competitors (No).

For example, Large stores without competitors average ZAR 279.30, whereas Large stores with nearby competitors drop to ZAR 267.81.

In [0]:
# Group by loyalty status to compare average basket value, item counts, and total transaction volume
loyalty_table = merged_df.groupby('is_loyalty_member').agg(
    total_revenue=('basket_value_zar', 'sum'),
    transaction_count=('transaction_id', 'count'),
    avg_basket_value=('basket_value_zar', 'mean'),
    avg_items=('num_items', 'mean')
).reset_index()

display(loyalty_table)

Transaction Counts & Volumes: Non-loyalty shoppers account for the higher share of footfall with 35,674 transactions (totalling ZAR 9,267,823.80), whereas loyalty members account for 28,656 transactions (totalling ZAR 8,428,290.20).

Basket Value Comparison: Loyalty members spend significantly more per visit, with an average basket value of ZAR 294.12 and 14.43 items per basket.

Non-Loyalty Shoppers: Non-loyalty shoppers have a notably smaller basket value, averaging ZAR 259.79 with 12.66 items per basket.

**2. Visualisation and Interpretation**

In [0]:
import plotly.express as px

# 1. Group by month and loyalty membership status
loyalty_trend = merged_df.groupby(['month', 'is_loyalty_member']).agg(
    avg_basket_value=('basket_value_zar', 'mean'),
    transaction_count=('transaction_id', 'count'),
    total_revenue=('basket_value_zar', 'sum')
).reset_index()


In [0]:
# 2. Plot monthly average basket value trend by loyalty status using Plotly Express
fig = px.line(
    loyalty_trend, 
    x='month', 
    y='avg_basket_value', 
    color='is_loyalty_member',
    markers=True,
    title="FreshMart 2025: Monthly Average Basket Value Trend (Loyalty vs. Non-Loyalty)",
    labels={'month': 'Month', 'avg_basket_value': 'Avg Basket Value (ZAR)', 'is_loyalty_member': 'Loyalty Member'}
)

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Average Basket Value (ZAR)"
)

fig.show()

In [0]:
#Merge transactions with stockouts on store_id and date
stockout_merged = transactions_df.merge(
    stockouts_df[['store_id', 'date', 'category', 'duration_hours']], 
    left_on=['store_id', 'transaction_date'], 
    right_on=['store_id', 'date'], 
    how='left'
)
display(stockout_merged.head(5))

In [0]:


# Plot average basket value by store format and competitor presence
fig_format = px.bar(
    format_competitor_segment,
    x='store_format',
    y='avg_basket_value',
    color='has_nearby_competitor',
    barmode='group',
    title="FreshMart 2025: Impact of Nearby Competitors on Basket Value by Store Format",
    labels={
        'store_format': 'Store Format',
        'avg_basket_value': 'Average Basket Value (ZAR)',
        'has_nearby_competitor': 'Nearby Competitor'
    },
    text_auto='.1f'
)

fig_format.update_layout(
    xaxis_title="Store Format",
    yaxis_title="Average Basket Value (ZAR)",
    template="plotly_white"
)

fig_format.show()

In [0]:
import plotly.express as px

# 1. Group by Province and Store Format for total revenue contribution
province_format_summary = merged_df.groupby(['province', 'store_format']).agg(
    total_revenue=('basket_value_zar', 'sum'),
    transaction_count=('transaction_id', 'count')
).reset_index()

# 2. Plot a clean bar chart using Plotly Express
fig_province = px.bar(
    province_format_summary,
    x='province',
    y='total_revenue',
    color='store_format',
    barmode='group',
    title="FreshMart 2025: Total Revenue Contribution by Province and Store Format",
    labels={
        'province': 'Province',
        'total_revenue': 'Total Revenue (ZAR)',
        'store_format': 'Store Format'
    },
    text_auto='.2s'
)

fig_province.update_layout(
    xaxis_title="Province",
    yaxis_title="Total Revenue (ZAR)",
    template="plotly_white"
)

fig_province.show()

Gauteng drives the largest share of revenue (especially in Large stores at 5.5M ZAR), followed by Western Cape and KwaZulu-Natal.